# Paper 1 — Matched Hold-out vs Five-Fold OOF Stacking

This cleaned notebook performs the **CT matched protocol comparison first**.

It is designed to:

- load the already saved five-fold OOF probabilities,
- load the already saved final test probabilities,
- avoid rerunning completed OOF folds and final full-development models,
- train only the missing fixed hold-out base models,
- train identical Logistic Regression and LightGBM meta-learners,
- compare hold-out stacking, OOF stacking and soft voting on the same untouched test set.

Run the cells from top to bottom.


## Cell 1 — Install dependencies and mount Google Drive


In [ ]:
!pip -q install lightgbm

from google.colab import drive
drive.mount("/content/drive")


## Cell 2 — Imports, reproducibility and paths

Change only the paths in this cell when your Google Drive folders differ.


In [ ]:
import os
import gc
import json
import time
import random
import pickle
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    log_loss,
    confusion_matrix,
    classification_report
)

from lightgbm import LGBMClassifier

from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint
)

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

# Existing CT data file
CT_DATA_PATH = (
    "/content/drive/MyDrive/"
    "PhD_CT_Transfer_Learning/data/"
    "ct_preprocessed.pkl"
)

# Existing completed OOF and final-test probability folder
CT_EXISTING_OUTPUT_DIR = (
    "/content/drive/MyDrive/"
    "CT_OOF_Stacking_Results"
)

CT_EXISTING_OOF_DIR = os.path.join(
    CT_EXISTING_OUTPUT_DIR,
    "OOF_Probabilities"
)

# New paper-specific folder
PAPER1_DIR = (
    "/content/drive/MyDrive/"
    "Paper_1_Matched_Protocol_Comparison"
)

CT_PAPER_DIR = os.path.join(PAPER1_DIR, "CT")
CT_SPLIT_DIR = os.path.join(CT_PAPER_DIR, "Split_Indices")
CT_HOLDOUT_DIR = os.path.join(CT_PAPER_DIR, "Holdout_Probabilities")
CT_MODEL_DIR = os.path.join(CT_PAPER_DIR, "Holdout_Models")
CT_RESULT_DIR = os.path.join(CT_PAPER_DIR, "Results")
CT_PREDICTION_DIR = os.path.join(CT_PAPER_DIR, "Test_Predictions")
CT_FIGURE_DIR = os.path.join(CT_PAPER_DIR, "Figures")

for directory in [
    PAPER1_DIR,
    CT_PAPER_DIR,
    CT_SPLIT_DIR,
    CT_HOLDOUT_DIR,
    CT_MODEL_DIR,
    CT_RESULT_DIR,
    CT_PREDICTION_DIR,
    CT_FIGURE_DIR
]:
    os.makedirs(directory, exist_ok=True)

print("TensorFlow:", tf.__version__)
print("Paper output:", PAPER1_DIR)
print("Existing OOF folder:", CT_EXISTING_OOF_DIR)


## Cell 3 — Load and validate the saved CT dataset

This uses the same `enhanced` preprocessing variant used in the corrected OOF experiment.


In [ ]:
if not os.path.exists(CT_DATA_PATH):
    raise FileNotFoundError(
        "CT data file not found. Update CT_DATA_PATH in Cell 2:\n"
        + CT_DATA_PATH
    )

with open(CT_DATA_PATH, "rb") as file:
    ct_data = pickle.load(file)

CT_VARIANT = "enhanced"

X_train_original = np.asarray(
    ct_data[f"X_train_{CT_VARIANT}"],
    dtype=np.float32
)

X_val_original = np.asarray(
    ct_data[f"X_val_{CT_VARIANT}"],
    dtype=np.float32
)

X_ct_test = np.asarray(
    ct_data[f"X_test_{CT_VARIANT}"],
    dtype=np.float32
)

y_train_original = np.asarray(
    ct_data["y_train"]
).reshape(-1).astype(np.int64)

y_val_original = np.asarray(
    ct_data["y_val"]
).reshape(-1).astype(np.int64)

y_ct_test = np.asarray(
    ct_data["y_test"]
).reshape(-1).astype(np.int64)

ct_class_names = list(ct_data["class_names"])

X_ct_development = np.concatenate(
    [X_train_original, X_val_original],
    axis=0
)

y_ct_development = np.concatenate(
    [y_train_original, y_val_original],
    axis=0
)

if X_ct_development.max() > 1.0:
    X_ct_development /= 255.0

if X_ct_test.max() > 1.0:
    X_ct_test /= 255.0

X_ct_development = np.clip(
    X_ct_development,
    0.0,
    1.0
).astype(np.float32)

X_ct_test = np.clip(
    X_ct_test,
    0.0,
    1.0
).astype(np.float32)

assert X_ct_development.shape[1:] == (224, 224, 1)
assert X_ct_test.shape[1:] == (224, 224, 1)
assert len(X_ct_development) == len(y_ct_development)
assert len(X_ct_test) == len(y_ct_test)

print("Development:", X_ct_development.shape)
print("Test:", X_ct_test.shape)
print("Development class counts:", np.bincount(y_ct_development))
print("Test class counts:", np.bincount(y_ct_test))
print("Classes:", ct_class_names)


## Cell 4 — Create or reload the fixed 80:20 hold-out split

The split is saved once and reused on every later run.


In [ ]:
base_indices_path = os.path.join(
    CT_SPLIT_DIR,
    "CT_Base_Train_Indices.npy"
)

meta_indices_path = os.path.join(
    CT_SPLIT_DIR,
    "CT_Meta_Holdout_Indices.npy"
)

all_indices = np.arange(len(y_ct_development))

if (
    os.path.exists(base_indices_path)
    and os.path.exists(meta_indices_path)
):
    ct_base_train_indices = np.load(base_indices_path)
    ct_meta_holdout_indices = np.load(meta_indices_path)
    print("Loaded the previously saved hold-out split.")
else:
    (
        ct_base_train_indices,
        ct_meta_holdout_indices
    ) = train_test_split(
        all_indices,
        test_size=0.20,
        stratify=y_ct_development,
        random_state=RANDOM_STATE
    )

    np.save(base_indices_path, ct_base_train_indices)
    np.save(meta_indices_path, ct_meta_holdout_indices)
    print("Created and saved a new hold-out split.")

X_ct_base = X_ct_development[ct_base_train_indices]
y_ct_base = y_ct_development[ct_base_train_indices]

X_ct_meta_holdout = X_ct_development[ct_meta_holdout_indices]
y_ct_meta_holdout = y_ct_development[ct_meta_holdout_indices]

print("Base-model subset:", X_ct_base.shape)
print("Meta-training hold-out:", X_ct_meta_holdout.shape)
print("Base class counts:", np.bincount(y_ct_base))
print("Meta class counts:", np.bincount(y_ct_meta_holdout))


## Cell 5 — CT augmentation and model builders

These builders match the corrected CT OOF implementations.


In [ ]:
CT_HEAD_LR = 1e-3
CT_FINE_TUNE_LR = 1e-5
CT_FINE_TUNE_LAYERS = 40

def make_ct_augmentation(name):
    return tf.keras.Sequential(
        [
            layers.RandomFlip("horizontal"),
            layers.RandomRotation(
                factor=0.03,
                fill_mode="reflect"
            ),
            layers.RandomTranslation(
                height_factor=0.03,
                width_factor=0.03,
                fill_mode="reflect"
            ),
            layers.RandomZoom(
                height_factor=(-0.05, 0.05),
                width_factor=(-0.05, 0.05),
                fill_mode="reflect"
            )
        ],
        name=name
    )

def build_ct_resnet50(trainable_backbone=False):
    inputs = layers.Input(
        shape=(224, 224, 1),
        name="ct_resnet_input"
    )

    x = make_ct_augmentation(
        "ct_resnet_augmentation"
    )(inputs)

    x = layers.Concatenate(
        axis=-1,
        name="grayscale_to_rgb"
    )([x, x, x])

    x = layers.Rescaling(
        scale=255.0,
        name="restore_255_scale"
    )(x)

    x = layers.Normalization(
        mean=[103.939, 116.779, 123.680],
        variance=[1.0, 1.0, 1.0],
        name="resnet50_mean_subtraction"
    )(x)

    backbone = tf.keras.applications.ResNet50(
        include_top=False,
        weights="imagenet",
        input_shape=(224, 224, 3)
    )

    backbone.trainable = trainable_backbone

    x = backbone(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)

    x = layers.Dense(
        256,
        activation="relu",
        kernel_regularizer=tf.keras.regularizers.l2(1e-4)
    )(x)

    x = layers.Dropout(0.50)(x)

    outputs = layers.Dense(
        2,
        activation="softmax",
        dtype="float32"
    )(x)

    model = models.Model(
        inputs,
        outputs,
        name="CT_ResNet50_Matched_Holdout"
    )

    model.compile(
        optimizer=optimizers.Adam(
            learning_rate=(
                CT_FINE_TUNE_LR
                if trainable_backbone
                else CT_HEAD_LR
            )
        ),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

def build_ct_efficientnetb0(trainable_backbone=False):
    inputs = layers.Input(
        shape=(224, 224, 1),
        name="ct_effnet_input"
    )

    x = make_ct_augmentation(
        "ct_effnet_augmentation"
    )(inputs)

    x = layers.Concatenate(
        axis=-1,
        name="grayscale_to_rgb"
    )([x, x, x])

    x = layers.Rescaling(
        scale=255.0,
        name="restore_255_scale"
    )(x)

    backbone = tf.keras.applications.EfficientNetB0(
        include_top=False,
        weights="imagenet",
        input_shape=(224, 224, 3)
    )

    backbone.trainable = trainable_backbone

    x = backbone(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)

    x = layers.Dense(
        256,
        activation="swish",
        kernel_regularizer=tf.keras.regularizers.l2(1e-4)
    )(x)

    x = layers.Dropout(0.50)(x)

    outputs = layers.Dense(
        2,
        activation="softmax",
        dtype="float32"
    )(x)

    model = models.Model(
        inputs,
        outputs,
        name="CT_EfficientNetB0_Matched_Holdout"
    )

    model.compile(
        optimizer=optimizers.Adam(
            learning_rate=(
                CT_FINE_TUNE_LR
                if trainable_backbone
                else CT_HEAD_LR
            )
        ),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model


## Cell 6 — Training helper for only the missing hold-out probabilities

The meta-holdout set is not used for early stopping.  
An internal validation subset is created only from the 80% base-training subset.


In [ ]:
def set_backbone_trainability(
    model,
    backbone_keyword,
    trainable_layers=40
):
    """
    Locate the nested Keras backbone model and unfreeze only its
    last trainable_layers layers, while keeping BatchNormalization frozen.
    """

    backbone = None

    # Only nested Keras models can be the application backbone.
    for layer in model.layers:
        if (
            isinstance(layer, tf.keras.Model)
            and backbone_keyword.lower() in layer.name.lower()
        ):
            backbone = layer
            break

    if backbone is None:
        nested_models = [
            layer.name
            for layer in model.layers
            if isinstance(layer, tf.keras.Model)
        ]

        raise ValueError(
            f"Could not locate backbone containing "
            f"'{backbone_keyword}'. "
            f"Nested models found: {nested_models}"
        )

    print("Backbone selected:", backbone.name)
    print("Total backbone layers:", len(backbone.layers))

    # First freeze the complete backbone.
    backbone.trainable = True

    for layer in backbone.layers:
        layer.trainable = False

    # Unfreeze only the final requested layers.
    for layer in backbone.layers[-trainable_layers:]:
        if not isinstance(
            layer,
            tf.keras.layers.BatchNormalization
        ):
            layer.trainable = True

    trainable_count = sum(
        int(layer.trainable)
        for layer in backbone.layers
    )

    print(
        "Trainable backbone layers:",
        trainable_count
    )

    return backbone


## Cell 7 — Generate or reload ResNet50 hold-out probabilities


In [ ]:
ct_resnet_holdout = generate_holdout_probabilities(
    model_name="ResNet50",
    builder=build_ct_resnet50,
    backbone_keyword="resnet50",
    probability_filename=(
        "CT_ResNet50_Holdout_Meta_Probabilities.npy"
    ),
    weights_filename=(
        "CT_ResNet50_Holdout.weights.h5"
    )
)

print(
    "ResNet50 hold-out shape:",
    ct_resnet_holdout.shape
)


## Cell 8 — Generate or reload EfficientNetB0 hold-out probabilities


In [ ]:
ct_effnet_holdout = generate_holdout_probabilities(
    model_name="EfficientNetB0",
    builder=build_ct_efficientnetb0,
    backbone_keyword="efficientnet",
    probability_filename=(
        "CT_EfficientNetB0_Holdout_Meta_Probabilities.npy"
    ),
    weights_filename=(
        "CT_EfficientNetB0_Holdout.weights.h5"
    )
)

print("EfficientNetB0 hold-out shape:", ct_effnet_holdout.shape)


## Cell 9 — Load all already completed OOF and final-test probability files

This cell must only load files. It does not retrain any OOF fold or final model.


In [ ]:
required_existing_files = {
    "ResNet50 OOF": os.path.join(
        CT_EXISTING_OOF_DIR,
        "CT_ResNet50_5Fold_OOF.npy"
    ),
    "EfficientNetB0 OOF": os.path.join(
        CT_EXISTING_OOF_DIR,
        "CT_EfficientNetB0_5Fold_OOF.npy"
    ),
    "ResNet50 test": os.path.join(
        CT_EXISTING_OOF_DIR,
        "CT_ResNet50_Test_Probabilities.npy"
    ),
    "EfficientNetB0 test": os.path.join(
        CT_EXISTING_OOF_DIR,
        "CT_EfficientNetB0_Test_Probabilities.npy"
    )
}

missing_files = [
    f"{name}: {path}"
    for name, path in required_existing_files.items()
    if not os.path.exists(path)
]

if missing_files:
    raise FileNotFoundError(
        "Saved completed files were not found. "
        "Update CT_EXISTING_OUTPUT_DIR in Cell 2.\n\n"
        + "\n".join(missing_files)
    )

ct_resnet_oof = np.load(
    required_existing_files["ResNet50 OOF"]
).astype(np.float32)

ct_effnet_oof = np.load(
    required_existing_files["EfficientNetB0 OOF"]
).astype(np.float32)

ct_resnet_test = np.load(
    required_existing_files["ResNet50 test"]
).astype(np.float32)

ct_effnet_test = np.load(
    required_existing_files["EfficientNetB0 test"]
).astype(np.float32)

expected_oof_shape = (
    len(y_ct_development),
    2
)

expected_test_shape = (
    len(y_ct_test),
    2
)

for name, array, expected_shape in [
    ("ResNet50 OOF", ct_resnet_oof, expected_oof_shape),
    ("EfficientNetB0 OOF", ct_effnet_oof, expected_oof_shape),
    ("ResNet50 test", ct_resnet_test, expected_test_shape),
    ("EfficientNetB0 test", ct_effnet_test, expected_test_shape)
]:
    if array.shape != expected_shape:
        raise ValueError(
            f"{name}: obtained {array.shape}, expected {expected_shape}"
        )

    if not np.isfinite(array).all():
        raise ValueError(f"{name} contains NaN or infinity.")

    print(name, array.shape)

print("\nAll completed OOF and final-test files were loaded.")


## Cell 10 — Build matched meta-training and meta-test matrices


In [ ]:
X_ct_holdout_meta = np.concatenate(
    [
        ct_resnet_holdout,
        ct_effnet_holdout
    ],
    axis=1
).astype(np.float32)

y_ct_holdout_meta = y_ct_meta_holdout.copy()

X_ct_oof_meta = np.concatenate(
    [
        ct_resnet_oof,
        ct_effnet_oof
    ],
    axis=1
).astype(np.float32)

y_ct_oof_meta = y_ct_development.copy()

X_ct_test_meta = np.concatenate(
    [
        ct_resnet_test,
        ct_effnet_test
    ],
    axis=1
).astype(np.float32)

print("Hold-out meta:", X_ct_holdout_meta.shape)
print("OOF meta:", X_ct_oof_meta.shape)
print("Test meta:", X_ct_test_meta.shape)

np.save(
    os.path.join(
        CT_HOLDOUT_DIR,
        "CT_Holdout_Meta_Features.npy"
    ),
    X_ct_holdout_meta
)

np.save(
    os.path.join(
        CT_HOLDOUT_DIR,
        "CT_Holdout_Meta_Labels.npy"
    ),
    y_ct_holdout_meta
)


## Cell 11 — Identical Logistic Regression and LightGBM definitions


In [ ]:
def build_logistic_meta_learner():
    return Pipeline(
        [
            ("scaler", StandardScaler()),
            (
                "classifier",
                LogisticRegression(
                    C=1.0,
                    penalty="l2",
                    solver="liblinear",
                    max_iter=5000,
                    random_state=RANDOM_STATE
                )
            )
        ]
    )

def build_lightgbm_meta_learner():
    return LGBMClassifier(
        objective="binary",
        n_estimators=150,
        learning_rate=0.03,
        num_leaves=7,
        max_depth=3,
        min_child_samples=30,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_alpha=0.2,
        reg_lambda=1.0,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbosity=-1
    )


## Cell 12 — Train both protocols and save test probabilities


In [ ]:
models_and_training_data = {
    "Holdout_LogisticRegression": (
        build_logistic_meta_learner(),
        X_ct_holdout_meta,
        y_ct_holdout_meta
    ),
    "Holdout_LightGBM": (
        build_lightgbm_meta_learner(),
        X_ct_holdout_meta,
        y_ct_holdout_meta
    ),
    "OOF_LogisticRegression": (
        build_logistic_meta_learner(),
        X_ct_oof_meta,
        y_ct_oof_meta
    ),
    "OOF_LightGBM": (
        build_lightgbm_meta_learner(),
        X_ct_oof_meta,
        y_ct_oof_meta
    )
}

meta_test_probabilities = {}

for method_name, (
    meta_model,
    X_train_meta,
    y_train_meta
) in models_and_training_data.items():

    model_path = os.path.join(
        CT_MODEL_DIR,
        method_name + ".joblib"
    )

    probability_path = os.path.join(
        CT_PREDICTION_DIR,
        method_name + "_Test_Probabilities.npy"
    )

    if (
        os.path.exists(model_path)
        and os.path.exists(probability_path)
    ):
        meta_model = joblib.load(model_path)
        test_probabilities = np.load(
            probability_path
        ).astype(np.float32)

        print(method_name, "loaded from saved files.")
    else:
        meta_model.fit(
            X_train_meta,
            y_train_meta
        )

        test_probabilities = meta_model.predict_proba(
            X_ct_test_meta
        ).astype(np.float32)

        joblib.dump(meta_model, model_path)
        np.save(probability_path, test_probabilities)

        print(method_name, "trained and saved.")

    meta_test_probabilities[
        method_name
    ] = test_probabilities

ct_soft_voting = (
    ct_resnet_test + ct_effnet_test
) / 2.0

np.save(
    os.path.join(
        CT_PREDICTION_DIR,
        "SoftVoting_Test_Probabilities.npy"
    ),
    ct_soft_voting
)


## Cell 13 — Evaluation function and final matched comparison table


In [ ]:
def evaluate_probabilities(
    method,
    protocol,
    probabilities,
    y_true
):
    predictions = np.argmax(
        probabilities,
        axis=1
    )

    positive_probability = probabilities[:, 1]

    return {
        "Dataset": "CT",
        "Protocol": protocol,
        "Method": method,
        "Accuracy": accuracy_score(
            y_true,
            predictions
        ),
        "Macro_Precision": precision_score(
            y_true,
            predictions,
            average="macro",
            zero_division=0
        ),
        "Macro_Recall": recall_score(
            y_true,
            predictions,
            average="macro",
            zero_division=0
        ),
        "Macro_F1": f1_score(
            y_true,
            predictions,
            average="macro",
            zero_division=0
        ),
        "ROC_AUC": roc_auc_score(
            y_true,
            positive_probability
        ),
        "Log_Loss": log_loss(
            y_true,
            probabilities,
            labels=[0, 1]
        ),
        "Seed": RANDOM_STATE
    }

result_rows = []

result_rows.append(
    evaluate_probabilities(
        method="Soft Voting",
        protocol="Common baseline",
        probabilities=ct_soft_voting,
        y_true=y_ct_test
    )
)

for key, probabilities in meta_test_probabilities.items():
    if key.startswith("Holdout_"):
        protocol = "Fixed hold-out"
        method = key.replace("Holdout_", "")
    else:
        protocol = "Five-fold OOF"
        method = key.replace("OOF_", "")

    result_rows.append(
        evaluate_probabilities(
            method=method,
            protocol=protocol,
            probabilities=probabilities,
            y_true=y_ct_test
        )
    )

ct_results = pd.DataFrame(result_rows)

metric_columns = [
    "Accuracy",
    "Macro_Precision",
    "Macro_Recall",
    "Macro_F1",
    "ROC_AUC",
    "Log_Loss"
]

display_table = ct_results.copy()

for column in metric_columns:
    display_table[column] = display_table[
        column
    ].round(6)

display(display_table)

results_path = os.path.join(
    CT_RESULT_DIR,
    "CT_Matched_Holdout_vs_OOF_Results.csv"
)

ct_results.to_csv(
    results_path,
    index=False
)

print("Saved:", results_path)


## Cell 14 — Save predicted labels and print classification reports


In [ ]:
all_probability_outputs = {
    "SoftVoting": ct_soft_voting,
    **meta_test_probabilities
}

for method_name, probabilities in all_probability_outputs.items():
    predicted_labels = np.argmax(
        probabilities,
        axis=1
    )

    np.save(
        os.path.join(
            CT_PREDICTION_DIR,
            method_name + "_Predicted_Labels.npy"
        ),
        predicted_labels
    )

    print("\n" + "=" * 72)
    print(method_name)
    print("=" * 72)

    print(
        classification_report(
            y_ct_test,
            predicted_labels,
            target_names=ct_class_names,
            digits=4,
            zero_division=0
        )
    )

np.save(
    os.path.join(
        CT_PREDICTION_DIR,
        "CT_Test_True_Labels.npy"
    ),
    y_ct_test
)


## Cell 15 — Final comparison chart


In [ ]:
plot_df = ct_results.copy()

label_map = {
    ("Common baseline", "Soft Voting"): "Soft Voting",
    ("Fixed hold-out", "LogisticRegression"): "Hold-out LR",
    ("Fixed hold-out", "LightGBM"): "Hold-out LGBM",
    ("Five-fold OOF", "LogisticRegression"): "OOF LR",
    ("Five-fold OOF", "LightGBM"): "OOF LGBM"
}

plot_df["Label"] = [
    label_map.get(
        (protocol, method),
        f"{protocol} {method}"
    )
    for protocol, method in zip(
        plot_df["Protocol"],
        plot_df["Method"]
    )
]

plot_df["Accuracy_Percent"] = (
    plot_df["Accuracy"] * 100
)

plt.figure(figsize=(10, 6))

bars = plt.bar(
    plot_df["Label"],
    plot_df["Accuracy_Percent"]
)

plt.ylabel("Accuracy (%)")
plt.xlabel("Ensemble method")
plt.title(
    "CT matched hold-out versus five-fold OOF stacking"
)

plt.xticks(
    rotation=0,
    ha="center"
)

for bar, value in zip(
    bars,
    plot_df["Accuracy_Percent"]
):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.08,
        f"{value:.2f}%",
        ha="center",
        va="bottom",
        fontsize=10,
        fontweight="bold"
    )

minimum_value = plot_df[
    "Accuracy_Percent"
].min()

maximum_value = plot_df[
    "Accuracy_Percent"
].max()

plt.ylim(
    max(0, minimum_value - 2),
    min(100, maximum_value + 1.5)
)

plt.grid(
    axis="y",
    linestyle="--",
    alpha=0.4
)

plt.tight_layout()

figure_path = os.path.join(
    CT_FIGURE_DIR,
    "CT_Matched_Holdout_vs_OOF_Accuracy.png"
)

plt.savefig(
    figure_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Saved:", figure_path)

# End of CT matched experiment

The notebook intentionally excludes:

- all completed five-fold OOF training cells,
- all completed final full-development model-training cells,
- duplicated model-verification cells,
- folder-search and inspection cells,
- old CIFAR-10 cells.

After this CT experiment is checked, a separate streamlined CIFAR-10 matched notebook should be prepared using the same structure and the four saved CIFAR OOF/test probability files.


In [ ]:
# ============================================================
# SOFTWARE ENVIRONMENT
# ============================================================

import sys
import platform
import numpy as np
import pandas as pd
import sklearn
import tensorflow as tf
import lightgbm
import matplotlib
import statsmodels

print("Python:", sys.version)
print("Platform:", platform.platform())
print("TensorFlow:", tf.__version__)
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("LightGBM:", lightgbm.__version__)
print("Matplotlib:", matplotlib.__version__)
print("statsmodels:", statsmodels.__version__)